# Session X: Lab — Building an Interactive Personal Budget Manager

**Goal:** Combine everything from Sessions I–IX into one complete application.

---

## Exercise 1: Define the Transaction Model

Create a `Transaction` class with:
- `INCOME_CATEGORIES` and `EXPENSE_CATEGORIES` class attributes
- `__init__`: amount, category, description, is_income (default False), auto-set date. Raise `ValueError` if the amount is not a positive, finite number (use `math.isfinite()`) or if the category is not in the list for that transaction type
- `to_dict()` and `@classmethod from_dict()` for serialization
- `__repr__` for readable output
- Type hints on all methods

### How to approach this code

A Transaction class is a blueprint for one record. The two category lists are shared class data, while `__init__` checks the amount and category before storing each object's values; `to_dict()` and `from_dict()` provide the two directions between objects and JSON-friendly dictionaries. `__repr__()` makes objects readable in debugging. The test checks that values survive a serialization round trip and that invalid data is rejected.

In [ ]:
import math
from datetime import datetime


class Transaction:
    INCOME_CATEGORIES = ["Salary", "Freelance", "Gift", "Other income"]
    EXPENSE_CATEGORIES = ["Food", "Transport", "Entertainment", "Housing", "Utilities", "Other"]

    def __init__(self, amount: float, category: str, description: str,
                 is_income: bool = False) -> None:
        # TODO: raise ValueError for a bad amount or category, then store the fields
        pass

    def to_dict(self) -> dict:
        # TODO
        pass

    @classmethod
    def from_dict(cls, data: dict) -> Transaction:
        # TODO
        pass

    def __repr__(self) -> str:
        # TODO
        pass


# Test
t1 = Transaction(25.50, "Food", "Lunch")
print(t1)
t2 = Transaction(3000, "Salary", "Monthly", is_income=True)
print(t2)
data = t1.to_dict()
t1_copy = Transaction.from_dict(data)
print(f"Round-trip: {t1.amount == t1_copy.amount}")
try:
    Transaction(-5, "Food", "Refund?")
except ValueError as error:
    print("Rejected:", error)
try:
    Transaction(10, "Salary", "Not an expense")
except ValueError as error:
    print("Rejected:", error)

---
## Exercise 2: Input Validation Functions

Write:
- `get_valid_amount(prompt)` — positive, finite number (reject `nan` and `inf` with `math.isfinite()`), catches ValueError
- `get_valid_category(categories)` — numbered menu, validates int input
- `get_valid_description()` — rejects empty strings

### How to approach this code

Validation first converts input text, then checks whether the converted value follows the app's rules. A retry loop lets users recover from nonnumeric amounts; separate checks reject zero, negative, non-finite (`nan`, `inf`) or out-of-range values. Reusable helpers keep those rules consistent across menu options.

In [ ]:
import math


def get_valid_amount(prompt: str) -> float:
    # TODO: while True + try/except; reject non-finite and non-positive values
    pass


def get_valid_category(categories: list[str]) -> str:
    # TODO: numbered menu + validation
    pass


def get_valid_description() -> str:
    # TODO: reject empty
    pass


# Uncomment to test:
# print(get_valid_amount("Amount: $"))

---
## Exercise 3: JSON Persistence

Write `save_transactions(transactions, filepath)` and `load_transactions(filepath)`.
- Save through a temporary file and `os.replace()`, so a failed write never destroys the old file.
- A missing file (`FileNotFoundError`) is a normal first run: return `[]`.
- A file that is not valid JSON (`JSONDecodeError`), or valid JSON that is not a list, is damaged: move it to `filepath + ".bak"`, print a warning and return `[]`.

### How to approach this code

The save and load functions make in-memory transactions persist between program runs. Writing to a temporary file and swapping it in with `os.replace()` keeps the old file intact if the write fails. `FileNotFoundError` means a first run, while a damaged file is set aside as `.bak` so it is never overwritten. The test uses temporary paths to avoid changing real data.

In [ ]:
import json
import os


def save_transactions(transactions: list[dict], filepath: str = "data/budget.json") -> None:
    # TODO: write a temporary file, then os.replace() it into place
    pass


def load_transactions(filepath: str = "data/budget.json") -> list[dict]:
    # TODO: missing file -> []; damaged or wrong shape -> move to .bak, warn, []
    pass


# Test
test_data = [{"amount": 50, "category": "Food", "description": "Groceries", "is_income": False, "date": "2026-10-01"}]
save_transactions(test_data, "/tmp/test_budget.json")
loaded = load_transactions("/tmp/test_budget.json")
print(loaded)
print(load_transactions("/tmp/nonexistent.json"))
with open("/tmp/damaged_budget.json", "w", encoding="utf-8") as f:
    f.write('{"amount": 5,')
print(load_transactions("/tmp/damaged_budget.json"))
print(os.path.exists("/tmp/damaged_budget.json.bak"))

---
## Exercise 4: CSV Export

Write `export_to_csv(transactions, filepath)` using csv.DictWriter. Create the folder if it is missing, open the file with `newline=""` and `encoding="utf-8"`, and handle an empty list.

### How to approach this code

CSV export maps each transaction dictionary to a row with a stable set of columns. `writeheader()` labels the fields and `writerows()` writes records; the empty-list check prevents a false success message. Reading the temporary export back verifies the actual output.

In [ ]:
import csv


def export_to_csv(transactions: list[dict], filepath: str = "data/export.csv") -> None:
    # TODO
    pass


# Test
test_data = [{"date":"2026-10-01","category":"Food","description":"Groceries","amount":50.0,"is_income":False}]
export_to_csv(test_data, "/tmp/test_export.csv")
with open("/tmp/test_export.csv", encoding="utf-8") as f:
    print(f.read())
export_to_csv([])

---
## Exercise 5: Summary Report

Write `generate_report(transactions)` that calculates income/expenses/balance, groups expenses by category and shows percentages. Handle an empty list, and print no "Expenses by Category" heading for an income-only list.

### How to approach this code

The report calculates income, expenses, balance, and totals by category from the same transaction list. Generator expressions select records before summing; a dictionary accumulates category totals. Print the category block only when there is at least one expense: then `expenses > 0`, so the percentage division is safe, and an income-only report has no empty heading.

In [ ]:
def generate_report(transactions: list[dict]) -> None:
    # TODO
    pass


test_data = [
    {"amount": 3000, "category": "Salary", "description": "Monthly", "is_income": True},
    {"amount": 800, "category": "Housing", "description": "Rent", "is_income": False},
    {"amount": 150, "category": "Food", "description": "Groceries", "is_income": False},
    {"amount": 75, "category": "Food", "description": "Restaurants", "is_income": False},
    {"amount": 50, "category": "Transport", "description": "Metro", "is_income": False},
]
generate_report(test_data)
generate_report([])
generate_report(test_data[:1])   # income only: no "Expenses by Category" heading

---
## Exercise 6: Main Application Loop

Wire everything together with a menu: Add Income, Add Expense, View All, Report, Export CSV, Save & Quit, Quit.
- At start-up, load the saved records and convert them with `clean_records()`, which skips (and reports) any record that fails validation.
- `ask_transaction(is_income)` asks for the amount, a category from the right list, and a description.
- Report, export and save work with dictionaries: convert with `[t.to_dict() for t in transactions]`.
- Q asks before discarding unsaved changes; any other key prints an error message.

### How to approach this code

The main loop coordinates the application: it displays choices and calls the relevant validation, model, reporting, export, and storage operations. Keep those implementations in their own functions so each menu branch remains easy to read. Saving before exit differs from quitting without saving, so Q should warn when there are unsaved changes.

In [ ]:
def clean_records(records: list[dict]) -> list[Transaction]:
    # TODO: from_dict() each record; report and skip the ones that raise an error
    pass


def ask_transaction(is_income: bool) -> Transaction:
    # TODO: amount, a category from the right list, description
    pass


def main() -> None:
    transactions = []  # TODO: load the saved records and clean them
    unsaved = False
    while True:
        print("\n=== PERSONAL BUDGET MANAGER ===")
        print("[1] Add Income  [2] Add Expense  [3] View All")
        print("[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit")
        choice = input("Select: ").strip().lower()
        if choice == "1":
            # TODO
            pass
        elif choice == "2":
            # TODO
            pass
        elif choice == "3":
            # TODO
            pass
        elif choice == "4":
            # TODO
            pass
        elif choice == "5":
            # TODO
            pass
        elif choice == "6":
            # TODO: save and break
            pass
        elif choice == "q":
            # TODO: if there are unsaved changes, ask before quitting
            print("Goodbye!")
            break
        else:
            print("Invalid option.")

# Uncomment to run (Ctrl-D or the stop button ends the input):
# main()